---
title: Introduction
---


This tutorial demonstrates how to use TVB-Optim to fit functional connectivity (FC) data from resting-state fMRI. We use the Reduced Wong-Wang (RWW) neural mass model to simulate brain activity, convert it to BOLD signal using a hemodynamic response function, and optimize model parameters to match empirical FC patterns.


In [ ]:
import os, importlib, subprocess, sys

def _available(*packages):
    return all(importlib.util.find_spec(p) for p in packages)

if not _available("tvbo", "tvboptim"):
    _devnull = open(os.devnull, "w")
    if "google.colab" in str(getattr(get_ipython(), "extension_manager", "")):
        subprocess.check_call(["pip", "install", "-q", "jax==0.6.1"], stdout=_devnull, stderr=_devnull)
        subprocess.check_call(["pip", "install", "-q", "--no-warn-conflicts", "tvbo", "tvboptim"], stdout=_devnull, stderr=_devnull)
    elif os.path.exists("/mnt/user"):
        script = os.path.join(os.path.dirname(os.path.abspath("__file__")), "setup_tvbo_tvboptim_env.sh")
        subprocess.check_call(["bash", script], stdout=_devnull, stderr=subprocess.STDOUT)
        print("Setup complete. Select the 'Python (tvb-o-ptim)' kernel and restart.")
    else:
        raise ImportError("tvbo and/or tvboptim not installed. Run: pip install tvbo tvboptim")
    _devnull.close()

The workflow includes:

- Building a whole-brain network with the RWW model
- Simulating BOLD signal from neural activity
- Computing functional connectivity from BOLD
- Optimizing global and region-specific parameters to fit target FC

In [ ]:
# Set up environment
cpu = True
if cpu:
    N_devices = 8
    os.environ['XLA_FLAGS'] = f'--xla_force_host_platform_device_count={N_devices}'

# Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import jax
import jax.numpy as jnp
import copy
import optax

from IPython.display import Markdown

# Import from tvboptim
from tvboptim.types import Parameter, Space, GridAxis
from tvboptim.types.stateutils import show_parameters
from tvboptim.execution import ParallelExecution
from tvboptim.optim.optax import OptaxOptimizer
from tvboptim.optim.callbacks import MultiCallback, DefaultPrintCallback, SavingCallback

# Network dynamics imports
from tvboptim.experimental.network_dynamics import prepare
from tvboptim.experimental.network_dynamics.solvers import Heun, BoundedSolver

# BOLD monitoring
from tvboptim.observations.tvb_monitors.bold import Bold

# Observation functions
from tvboptim.observations.observation import compute_fc, fc_corr, rmse

We enable 64-bit precision to get reliable gradient information.

In [ ]:
jax.config.update("jax_enable_x64", True)

## Loading Structural Data and Target FC

We load the Lobar8 parcellation (8 lobar regions per hemisphere, 16 total) structural connectivity and empirical functional connectivity from HCP Young Adults dTOR tractography.

In [ ]:
from tvbo import Network

lobar8 = Network.from_db(
    atlas="Lobar8",
    rec="avgMatrix",
)
lobar8.plot_overview(log_weights=True)

In [ ]:
lobar8

In [ ]:
print("min", np.min(lobar8.matrix("weight")), "max", np.max(lobar8.matrix("weight")))

In [ ]:
# Normalize weights to [0, 1] range
lobar8.normalize_weights()  # lobar8.transforms = {rhs: 'W / max(W)'}

lobar8.transforms

In [ ]:
print("min", np.min(lobar8.matrix("weight")), "max", np.max(lobar8.matrix("weight")))

In [ ]:
lobar8.node_labels

## The Reduced Wong-Wang Model

In [ ]:
from tvbo import Dynamics
dynamics = Dynamics.from_db("ReducedWongWangTvboptim")

Markdown(dynamics.render('report'))

In [ ]:
Markdown(f"```python\n{dynamics.render('tvboptim')}\n```")

In [ ]:
from tvbo import Noise

dynamics.state_variables["S"].noise = Noise(
    additive=True, parameters={"sigma": {"value": 0.01}}
)

## Building the Network Model

We combine the RWW dynamics with structural connectivity to create a whole-brain network model.

In [ ]:
lobar8.dynamics['ReducedWongWangTvboptim'] = dynamics
lobar8.coupling['instant'] = Coupling.from_db("FastLinearCoupling")
network = lobar8.execute('tvboptim')

In [ ]:
lobar8.coupling['instant'].delayed

In [ ]:
network.graph

## Preparing and Running the Simulation

We prepare the network for simulation and run an initial transient to reach a quasi-stationary state.

In [ ]:
# Prepare simulation: compile model and initialize state
t1 = 100_000  # Total simulation duration (ms) - 2 minutes
dt = 4.0      # Integration timestep (ms)
solver = BoundedSolver(Heun(), low=0.0, high=1.0)  # Bound S to [0, 1]
model, state = prepare(network, solver, t1=t1, dt=dt)

# First simulation: run transient to reach quasi-stationary state
result_init = model(state)

# Update network with final state as new initial conditions
network.update_history(result_init)
model, state = prepare(network, solver, t1=t1, dt=dt)

# Second simulation: quasi-stationary dynamics
result = model(state)

## Computing BOLD Signal

We convert the neural activity (synaptic gating S) to simulated BOLD signal using a hemodynamic response function. The BOLD monitor downsamples the neural activity and convolves it with a canonical HRF kernel.

In [ ]:
# Create BOLD monitor with standard parameters
bold_monitor = Bold(
    period=1000.0,          # BOLD sampling period (1 TR = 1000 ms)
    downsample_period=4.0,  # Intermediate downsampling matches dt
    voi=0,                  # Monitor first state variable (S)
    history=result_init     # Use initial state as warm start
)

# Apply BOLD monitor to simulation result
bold_result = bold_monitor(result)

In [ ]:
from matplotlib.colors import Normalize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.1, 3.0375))

# Plot raw neural activity (first 1000 ms)
t_max_idx = int(1000 / dt)
time_raw = result.time[:t_max_idx]
data_raw = result.data[:t_max_idx, 0, :]

num_lines = data_raw.shape[1]
cmap = plt.cm.cividis
mean_values = np.mean(data_raw, axis=0)
norm = Normalize(vmin=np.min(mean_values), vmax=np.max(mean_values))
for i in range(num_lines):
    color = cmap(norm(mean_values[i]))
    ax1.plot(time_raw, data_raw[:, i], color=color, linewidth=0.5)

ax1.text(0.95, 0.95, "Raw Neural Activity", transform=ax1.transAxes, fontsize=10,
         ha='right', va='top', bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
ax1.set_xlabel("Time [ms]")
ax1.set_ylabel("S [a.u.]")

# Plot BOLD signal (first 60 TRs)
t_bold_max = 60
time_bold = bold_result.time[:t_bold_max]
data_bold = bold_result.data[:t_bold_max, 0, :]

num_lines = data_bold.shape[1]
mean_values = np.mean(data_bold, axis=0)
norm = Normalize(vmin=np.min(mean_values), vmax=np.max(mean_values))
for i in range(num_lines):
    color = cmap(norm(mean_values[i]))
    ax2.plot(time_bold, data_bold[:, i], color=color, linewidth=0.8)

ax2.text(0.95, 0.95, "BOLD Signal", transform=ax2.transAxes, fontsize=10,
         ha='right', va='top', bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("BOLD [a.u.]")

plt.tight_layout()

## Defining Observations and Loss

Functional connectivity (FC) measures the temporal correlation between BOLD signals from different brain regions. We define an observation function that simulates BOLD and computes FC, and a loss function that quantifies the mismatch with empirical FC.

In [ ]:
fc_target = lobar8.matrix('fc')

In [ ]:
def observation(state):
    """Compute functional connectivity from simulated BOLD signal."""
    # Run simulation
    result = model(state)
    # Convert to BOLD
    bold = bold_monitor(result)
    # Compute FC, skipping first 20 TRs to avoid transient effects
    fc = compute_fc(bold, skip_t=20)
    return fc

def loss(state):
    """Compute RMSE between simulated and empirical FC."""
    fc = observation(state)
    # return 1- fc_corr(fc, fc_target)
    return rmse(fc, fc_target)

In [ ]:
# Calculate initial FC
fc_initial = np.array(observation(state))

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.1, 3.54375))

# Plot both FC matrices
for ax_current, fc_matrix, title_prefix in zip([ax1, ax2], [fc_target, fc_initial], ["Target FC", "Initial FC"]):
    fc_matrix = np.copy(fc_matrix)
    np.fill_diagonal(fc_matrix, np.nan)  # Set diagonal to NaN
    im = ax_current.imshow(fc_matrix, cmap='cividis', vmax=0.9)

    ax_current.set_xticks([])
    ax_current.set_yticks([])
    ax_current.set_xlabel('')
    ax_current.set_ylabel('')

    # Calculate correlation and RMSE for title
    if title_prefix == "Initial FC":
        corr_value = fc_corr(fc_initial, fc_target)
        rmse_value = rmse(fc_initial, fc_target)
        title = f"{title_prefix}\nr = {corr_value:.3f}, RMSE = {rmse_value:.3f}"
    else:
        title = title_prefix

    # Add title as annotation
    ax_current.annotate(title,
                       xy=(0.25, 0.95),
                       xycoords='axes fraction',
                       ha='left', va='top',
                       fontsize=10, fontweight='bold',
                       color='black',
                       bbox=dict(boxstyle='round,pad=0.3',
                                facecolor='white', alpha=0.9))

plt.tight_layout()

## Parameter Exploration

Before optimization, we explore how the model parameters affect FC quality. We systematically vary the excitatory recurrence `w` and global coupling strength `G` across a 2D grid and compute the loss for each combination.

In [ ]:
# Create grid for parameter exploration
n = 32

# Set up parameter axes for exploration
grid_state = copy.deepcopy(state)
grid_state.dynamics.w = GridAxis(0.001, 1.0, n)
grid_state.coupling.instant.G = GridAxis(0.001, 2.0, n)

# Create space (product creates all combinations of w and G)
grid = Space(grid_state, mode="product")

# Parallel execution across 8 processes
exec = ParallelExecution(loss, grid, n_pmap=8)
# Alternative: Sequential execution
# exec = SequentialExecution(loss, grid)
exploration_results = exec.run()

In [ ]:
# Prepare data for visualization
pc = grid.collect()
G_vals = pc.coupling.instant.G.flatten()
w_vals = pc.dynamics.w.flatten()

# Get parameter ranges
G_min, G_max = min(G_vals), max(G_vals)
w_min, w_max = min(w_vals), max(w_vals)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8.1, 4.05))

# Create the heatmap
im = ax.imshow(jnp.stack(exploration_results).reshape(n, n).T,
              cmap='cividis_r',
              extent=[G_min, G_max, w_min, w_max],
              origin='lower',
              aspect='auto',
              interpolation='hamming')

# Add colorbar and labels
cbar = plt.colorbar(im, label="Loss (RMSE)")
# cbar = plt.colorbar(im, label="Loss (Fc Correlation)")
ax.set_xlabel('Global Coupling (G)')
ax.set_ylabel('Excitatory Recurrence (w)')
ax.set_title("Parameter Exploration")

plt.tight_layout()

## Gradient-Based Optimization

We use gradient-based optimization to find the best global parameters (same values for all regions) that minimize the FC mismatch. JAX's automatic differentiation computes gradients through the entire simulation pipeline.

In [ ]:
import gc
gc.collect()

In [ ]:
# Mark parameters as optimizable
state.coupling.instant.G = Parameter(state.coupling.instant.G)
state.dynamics.w = Parameter(state.dynamics.w)

# Create and run optimizer
cb = MultiCallback([
    DefaultPrintCallback(every=10),
    SavingCallback(key="state", save_fun=lambda *args: args[1])  # Save updated state
])

opt = OptaxOptimizer(loss, optax.adam(0.01), callback=cb)
fitted_state, fitting_data = opt.run(state, max_steps=200)

In [ ]:
# Prepare data for visualization
pc = grid.collect()
G_vals = pc.coupling.instant.G
w_vals = pc.dynamics.w

# Get parameter ranges
G_min, G_max = min(G_vals), max(G_vals)
w_min, w_max = min(w_vals), max(w_vals)

# Create figure and axis
fig, ax = plt.subplots(figsize=(8.1, 4.725))

# Create the heatmap
im = ax.imshow(jnp.stack(exploration_results).reshape(n, n).T,
               cmap='cividis_r',
               extent=[G_min, G_max, w_min, w_max],
               origin='lower',
               aspect='auto',
               interpolation='hamming')

# Mark initial value
G_init = state.coupling.instant.G.value
w_init = state.dynamics.w.value
ax.scatter(G_init, w_init, color='white', s=100, marker='o',
           edgecolors='k', linewidths=2, zorder=5)

# Add annotation
ax.annotate('Initial', xy=(G_init, w_init),
            xytext=(G_init, w_init+0.05*(w_max-w_min)),
            color='white', fontweight='bold', ha='center', zorder=5,
            path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

# Add fitted value point
G_fit = fitted_state.coupling.instant.G.value
w_fit = fitted_state.dynamics.w.value
ax.scatter(G_fit, w_fit, color='white', s=100, marker='o',
           edgecolors='k', linewidths=2, zorder=5)

# Add annotation for the fitted value
ax.annotate('Optimized', xy=(G_fit, w_fit),
            xytext=(G_fit, w_fit-0.08*(w_max-w_min)),
            color='white', fontweight='bold', ha='center', zorder=5,
            path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

# Add optimization path points
G_route = np.array([ds.coupling.instant.G.value for ds in fitting_data["state"].save])
w_route = np.array([ds.dynamics.w.value for ds in fitting_data["state"].save])
ax.scatter(G_route[::2], w_route[::2], color='white', s=15, marker='o',
           linewidths=1, zorder=4, edgecolors='k')

# Remove axes ticks and labels
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()

## Heterogeneous Optimization

Global parameters (same for all regions) may not capture region-specific variations needed for optimal FC fit. We now make parameters heterogeneous: each brain region gets its own `w` and `I_o` values, while keeping `G` global.

In [ ]:
n_nodes = lobar8.number_of_nodes

# Copy already optimized state and make parameters regional
fitted_state_het = copy.deepcopy(fitted_state)

# # Make w regional (one value per node)
fitted_state_het.dynamics.w.shape = (n_nodes,)

# Also make I_o regional and mark as optimizable
fitted_state_het.dynamics.I_o = Parameter(fitted_state_het.dynamics.I_o)
fitted_state_het.dynamics.I_o.shape = (n_nodes,)

# Keep global coupling fixed at optimized value
fitted_state_het.coupling.instant.G = fitted_state_het.coupling.instant.G.value

# fitted_state_het.graph._weights = Parameter(fitted_state_het.graph.weights)
show_parameters(fitted_state_het)

In [ ]:
opt_het = OptaxOptimizer(loss, optax.adam(0.001), callback=cb)
fitted_state_het, fitting_data_het = opt_het.run(fitted_state_het, max_steps=200)

## Comparing Global vs Regional Parameters

Let's compare the FC quality from global (homogeneous) vs regional (heterogeneous) parameter fits.

In [ ]:
# Compute FC for both optimization strategies
fc_global = np.array(observation(fitted_state))
fc_regional = np.array(observation(fitted_state_het))

In [ ]:
# Create 2x2 figure: FC heatmaps with Target for visual comparison
fig, axes = plt.subplots(2, 2, figsize=(8.1, 8.1))

fc_list = [fc_target, fc_initial, fc_global, fc_regional]
fc_labels = ["Target FC", "Initial FC", "Global Parameters", "Regional Parameters"]

# Compute shared color range across all FC matrices (excluding diagonal)
all_offdiag = np.concatenate([fc[np.triu_indices_from(fc, k=1)] for fc in fc_list])
vmin_shared, vmax_shared = float(np.min(all_offdiag)), float(np.max(all_offdiag))
vmin_shared, vmax_shared = 0, 0.75

for ax_current, fc_matrix, label in zip(axes.flat, fc_list, fc_labels):
    fc_plot = np.copy(fc_matrix)
    np.fill_diagonal(fc_plot, np.nan)
    im = ax_current.imshow(fc_plot, cmap='cividis', vmin=vmin_shared, vmax=vmax_shared)
    ax_current.set_xticks([])
    ax_current.set_yticks([])

    if label == "Target FC":
        title = label
    else:
        corr_value = fc_corr(fc_matrix, fc_target)
        rmse_value = rmse(fc_matrix, fc_target)
        title = f"{label}\nr = {corr_value:.3f}, RMSE = {rmse_value:.3f}"

    ax_current.set_title(title, fontsize=10, fontweight='bold')

plt.tight_layout()

In [ ]:
# Scatter plots: simulated vs empirical FC
triu_idx = np.triu_indices_from(fc_target, k=1)
fc_target_triu = fc_target[triu_idx]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8.1, 3.24), sharey=True, sharex=True)

for ax_current, fc_matrix, label in zip([ax1, ax2, ax3], [fc_initial, fc_global, fc_regional], ["Initial", "Global Parameters", "Regional Parameters"]):
    fc_triu = fc_matrix[triu_idx]
    corr_value = fc_corr(fc_matrix, fc_target)
    rmse_value = rmse(fc_matrix, fc_target)
    ax_current.scatter(fc_target_triu, fc_triu, alpha=0.3, s=10, color='royalblue', edgecolors='none')
    ax_current.plot([fc_target_triu.min(), fc_target_triu.max()],
                    [fc_target_triu.min(), fc_target_triu.max()],
                    'k--', linewidth=1.5)
    ax_current.set_xlabel('Empirical FC')
    ax_current.set_title(f'{label}\nr = {corr_value:.3f}, RMSE = {rmse_value:.3f}')
    ax_current.grid(True, alpha=0.3)
    ax_current.set_aspect('equal', adjustable='box')

ax1.set_ylabel('Simulated FC')

plt.tight_layout()

## Fitted Heterogeneous Parameters

Let's examine the fitted region-specific parameters and their relationship to structural connectivity.

In [ ]:
# Calculate mean incoming connectivity for each region
mean_connectivity = np.mean(lobar8.matrix('weight'), axis=1)

# Extract fitted regional parameters
w_fitted = fitted_state_het.dynamics.w.value.flatten()
I_o_fitted = fitted_state_het.dynamics.I_o.value.flatten()

# Get global optimization values for reference
w_global = fitted_state.dynamics.w.value
I_o_global = fitted_state.dynamics.I_o  # Not optimized in global fit, but initial value

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.1, 3.24))

# Plot w vs mean connectivity
ax1.scatter(mean_connectivity, w_fitted, alpha=0.7, s=30, color='royalblue', edgecolors='k', linewidths=0.5)
ax1.axhline(w_global, color='red', linestyle='--', linewidth=2, label=f'Global w = {w_global:.3f}')
ax1.set_xlabel('Mean Incoming Connectivity')
ax1.set_ylabel('Fitted w (Excitatory Recurrence)')
ax1.set_title('Regional Excitatory Recurrence Parameters')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot I_o vs mean connectivity
ax2.scatter(mean_connectivity, I_o_fitted, alpha=0.7, s=30, color='royalblue', edgecolors='k', linewidths=0.5)
ax2.axhline(I_o_global, color='red', linestyle='--', linewidth=2, label=f'Initial I_o = {I_o_global:.3f}')
ax2.set_xlabel('Mean Incoming Connectivity')
ax2.set_ylabel('Fitted I_o (External Input)')
ax2.set_title('Regional External Input Parameters')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()

In [ ]:
from tvbo.classes.network import Network
from bsplot.surface import plot_surf

# Load per-hemisphere Lobar8 surface networks (fsaverage 164k)
surf_lh = Network.from_db(atlas="Lobar8", desc="surf", hemi="L")
surf_rh = Network.from_db(atlas="Lobar8", desc="surf", hemi="R")
rm_lh = surf_lh.node_mapping_data  # (163842,) values 0-5
rm_rh = surf_rh.node_mapping_data  # (163842,) values 8-13

# Extract fitted regional params (16 nodes)
w_fitted = np.array(fitted_state_het.dynamics.w.value).flatten()
I_o_fitted = np.array(fitted_state_het.dynamics.I_o.value).flatten()


# Project node values → vertex arrays per hemisphere (0 for medial wall)
def to_vertex(values, rm):
    out = np.full(len(rm), np.nan)
    valid = rm >= 0
    out[valid] = values[rm[valid]]
    return out


lh_w = to_vertex(w_fitted, rm_lh)
rh_w = to_vertex(w_fitted, rm_rh)
lh_Io = to_vertex(I_o_fitted, rm_lh)
rh_Io = to_vertex(I_o_fitted, rm_rh)

# Plot: 2 rows (w, I_o) × 5 cols (LH lat, LH med, RH lat, RH med, colorbar)
from matplotlib.colors import Normalize
import matplotlib.cm as mcm

fig = plt.figure(figsize=(9, 4.05), dpi=200, layout='compressed')
# 5 cols: 4 brain views + 1 narrow colorbar per row
gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 1, 0.07], wspace=0.05, hspace=0.1)

params = [("w", lh_w, rh_w, "cividis"), ("I_o", lh_Io, rh_Io, "cividis")]

for row, (name, lh_data, rh_data, cmap_name) in enumerate(params):
    combined = np.concatenate([lh_data[~np.isnan(lh_data)], rh_data[~np.isnan(rh_data)]])
    norm = Normalize(vmin=combined.min(), vmax=combined.max())

    for col, (hemi, data, view) in enumerate(
        [
            ("lh", lh_data, "lateral"),
            ("lh", lh_data, "medial"),
            ("rh", rh_data, "lateral"),
            ("rh", rh_data, "medial"),
        ]
    ):
        ax = fig.add_subplot(gs[row, col])
        plot_surf(
            "fsaverage",
            overlay=data,
            ax=ax,
            hemi=hemi,
            view=view,
            cmap=cmap_name,
            norm=norm,
        )
        ax.axis("off")
        if row == 0:
            ax.set_title(f"{hemi.upper()} {view}", fontsize=7)

    # Colorbar in 5th column
    cax = fig.add_subplot(gs[row, 4])
    sm = mcm.ScalarMappable(cmap=cmap_name, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, shrink=0.7)
    cb.ax.tick_params(labelsize=6)
    cb.set_label(name, fontsize=8)

plt.savefig("brain_params.png", bbox_inches="tight", dpi=200)

plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3))
colors = ["steelblue" if "LH" in l else "tomato" for l in dk.node_labels]
for ax, (vals, name) in zip(axs, [(w_fitted, "w"), (I_o_fitted, "I_o")]):
    ax.bar(dk.node_labels, vals, color=colors)
    ax.set_xticklabels(dk.node_labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(name)
    ax.axhline(vals.mean(), color="k", linestyle="--", linewidth=1)
plt.tight_layout()

In [ ]:
from tvbo import SimulationExperiment

yaml_definition = """
id: 2
label: "RWW Functional Connectivity Fitting"
description: >
    Fitting functional connectivity using the Reduced Wong-Wang model.
    Two-stage optimization: global parameters first, then regional heterogeneity.

dynamics:
    name: ReducedWongWang
    label: "Reduced Wong-Wang Model"
    output: [S, H]

    parameters:
        a:  {value: 0.270, unit: per_nC}
        b:  {value: 0.108, unit: kHz}
        d:  {value: 154.0, unit: ms}
        gamma: {value: 0.641}
        tau_s: {value: 100.0, unit: ms}
        w:  {value: 0.5, domain: {lo: 0.0, hi: 1.0}}
        J_N: {value: 0.2609}
        I_o: {value: 0.32}

    derived_variables:
        x:
            equation:
                rhs: "w * J_N * S + I_o + J_N * FastLinearCoupling"
        H:
            equation:
                rhs: "(a * x - b) / (1 - exp(-d * (a * x - b)))"

    state_variables:
        S:
            equation:
                rhs: "-(S / tau_s) + (1 - S) * H * gamma"
            initial_value: 0.3
            domain: {lo: 0.0, hi: 1.0}
            coupling_variable: true
            noise:
                additive: true
                parameters:
                    sigma: {value: 0.00283}

    coupling_inputs:
        FastLinearCoupling: {dimension: 1}

network:
    label: "Desikan-Killiany"
    bids_dir: "../networks/bids/dk_average"
    structural_measures: [streamlineCount, tractLength]
    observational_measures: [BoldCorrelation]
    transforms:
        - name: weight
          equation: {rhs: "W / max(W)"}
    coupling:
        FastLinearCoupling:
            label: "Fast Linear Coupling"
            delayed: false
            parameters:
                G: {value: 0.5, free: true, domain: {lo: 0.0, hi: 1.0}}
            local_states: [S]
            pre_expression:  {rhs: "local_states"}
            post_expression: {rhs: "G * gx"}

integration:
    method: Heun
    step_size: 4.0
    duration: 120000
    transient_time: 120000

functions:
    temporal_average:
        source_code: "jnp.mean(data.reshape(-1, period_samples, *data.shape[1:]), axis=1)"
        arguments:
            - {name: data}
            - {name: period_samples, value: 1}

    hrf_kernel:
        time_range: {lo: 0, hi: duration, n: 5000}
        equation:
            rhs: "(1/3)*exp(-0.5*(t/1000)/tau_s)*sin(sqrt(1/tau_f - 1/(4*tau_s**2))*(t/1000))/sqrt(1/tau_f - 1/(4*tau_s**2))"
            parameters:
                tau_s: {value: 0.8}
                tau_f: {value: 0.4}
        arguments:
            - {name: duration, value: 20000.0}

    prepend_history:
        source_code: "jnp.concatenate([history[-kernel_samples:], data], axis=0)"
        arguments:
            - {name: data}
            - {name: history}
            - {name: kernel_samples, value: 5000}

    volterra_transform:
        equation:
            rhs: "k_1 * V_0 * (data - 1.0)"
            parameters:
                k_1: {value: 5.6}
                V_0: {value: 0.02}
        arguments:
            - {name: data}

    subsample_bold:
        source_code: "data[::step][:n_samples]"
        arguments:
            - {name: data}
            - {name: step, value: 250}
            - {name: n_samples, value: 120}

    rmse:
        equation: {rhs: "sqrt(mean((x - y)**2))"}
        arguments:
            - {name: x}
            - {name: y}

    fc_corr:
        callable: {name: fc_corr, module: tvboptim.observations.observation}

observations:
    empirical_fc:
        label: "Empirical Functional Connectivity"
        source: network.observations.BoldCorrelation

    bold:
        label: "Simulated BOLD Signal"
        source: S
        period: 1000.0
        pipeline:
            - function: hrf_kernel

            - output: downsampled_data
              function: temporal_average
              arguments:
                  - {name: data, value: integration.result}
                  - {name: period_samples, value: 1}

            - output: downsampled_history
              function: temporal_average
              arguments:
                  - {name: data, value: integration.transient}
                  - {name: period_samples, value: 1}

            - function: prepend_history
              arguments:
                  - {name: data, value: downsampled_data}
                  - {name: history, value: downsampled_history}
                  - {name: kernel_samples, value: 5000}

            - output: bold_convolve
              callable: {module: jax.scipy.signal, name: fftconvolve}
              apply_on_dimension: node
              arguments:
                  - {name: in1, value: prepend_history}
                  - {name: in2, value: hrf_kernel}
                  - {name: mode, value: valid}

            - function: volterra_transform
              arguments:
                  - {name: data, value: bold_convolve}

            - function: subsample_bold
              arguments:
                  - {name: data, value: volterra_transform}

derived_observations:
    fc:
        label: "Simulated Functional Connectivity"
        source_observations: [bold]
        pipeline:
            - callable: {name: compute_fc, module: tvboptim.observations.observation}
              arguments:
                  - {name: timeseries, value: bold}
                  - {name: skip_t, value: 20}

optimizations:
    fc_fitting:
        label: "FC Fitting"
        description: >
            Two-stage optimization: global (w, G) then regional (w, I_o).

        loss:
            function: rmse
            arguments:
                - {name: simulated_fc, value: observations.fc.data}
                - {name: empirical_fc, value: observations.empirical_fc.data}

        stages:
            - name: global_optimization
              label: "Global Parameter Optimization"
              free_parameters:
                  - ReducedWongWang.w
                  - FastLinearCoupling.G
              algorithm: adam
              learning_rate: 0.01
              max_iterations: 300
              hyperparameters:
                  - {name: b2, value: 0.9999}

            - name: regional_optimization
              label: "Regional Parameter Optimization"
              warmup_from: global_optimization
              free_parameters:
                  - {name: ReducedWongWang.w, heterogeneous: true}
                  - {name: ReducedWongWang.I_o, heterogeneous: true}
              algorithm: adam
              learning_rate: 0.004
              max_iterations: 200
              hyperparameters:
                  - {name: b2, value: 0.999}

explorations:
    parameter_landscape:
        label: "w-G Parameter Landscape"
        parameters:
            - name: ReducedWongWang.w
              domain: {lo: 0.001, hi: 0.7, n: 32}
            - name: FastLinearCoupling.G
              domain: {lo: 0.001, hi: 0.7, n: 32}
        mode: product
        observable:
            function: rmse
            arguments:
                - {name: simulated_fc, value: fc.data}
                - {name: empirical_fc, value: empirical_fc.data}

execution:
    n_workers: 8
    precision: float64
    random_seed: 0
"""

exp = SimulationExperiment.from_string(yaml_definition)
results = exp.run("tvboptim")